In [ ]:
%load_ext autoreload
%autoreload 2

import ollama
import chromadb
from pyprojroot import here
from odyssey import OllamaEmbeddings
from pydantic import BaseModel, Field

In [ ]:
embed_model = "qwen3-embedding:4b"
embed_fn = OllamaEmbeddings(model=embed_model)

In [ ]:
client = chromadb.PersistentClient(path=here("db/chroma"))
collection = client.get_collection("books", embed_fn)

In [ ]:
query = "Who is Odysseus son?"


result = collection.query(
    query_texts=[query],
    n_results=5
)

print(result["distances"])

In [ ]:
prompt = """<query>{query}</query>"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "user", "content": prompt.format(query=query)}
    ]
)

print(response.message.content)

In [ ]:
prompt = """<query>{query}</query><context>{context}</context>"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "user", "content": prompt.format(query=query, context=result["documents"])}
    ]
)

print(response.message.content)

In [ ]:
prompt = """Determine how relevant the given contex is to the users query. Give a score from 0% to 100%. 
<query>{query}</query>
<context>{context}</context>
"""

class RelevenceScore(BaseModel):
    score: int
    reason: str = Field("Reasoning for the given relevence score.")


response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "user", "content": prompt.format(query=query, context=result["documents"])}
    ],
    format=RelevenceScore.model_json_schema()
)

RelevenceScore.model_validate_json(response.message.content)